In [ ]:
import pandas as pd # Data structure for CSV
import re # Regular Expressions
from sklearn.feature_selection import VarianceThreshold
import numpy as np # Numpy need no introduction

df = pd.read_csv('../dataset/full.csv')
prev_df = pd.read_csv('../dataset/full.csv')
df_columns = df.columns.tolist()
df_cleaned = pd.DataFrame()
prev_df_columns = []

df.head()

In [ ]:
def compare_df():    
    if df_cleaned.empty:
        return
    df_cleaned_columns = df_cleaned.columns.tolist()
    prev_df_columns = prev_df.columns.tolist()

    diff_list_total = list(set(df_columns) - set(df_cleaned_columns))
    diff_list = list(list(set(prev_df_columns) - set(df_cleaned_columns)))

    df_columns_count = len(df_columns)
    df_cleaned_columns_count = len(df_cleaned_columns)
    prev_df_columns_count = len(prev_df_columns)

    total_removed_features = df_columns_count - df_cleaned_columns_count
    this_step_removed_features = prev_df_columns_count - df_cleaned_columns_count

    print("Features excluded (in total):")
    print(diff_list_total)
    print('='*50)
    print("Features excluded (this step):")
    print(diff_list)
    print('='*50)
    print("Removed total: " + str(df_columns_count) + " ---> " + str(df_cleaned_columns_count) + " features")
    print("Total feature removal count: " + str(total_removed_features))
    print('='*50)
    print("Removed this step: " + str(prev_df_columns_count) + " ---> " + str(df_cleaned_columns_count) + " features")
    print("Feature removal this step count: " + str(this_step_removed_features))
    


# Feature extraction/selection

First, we remove features that contain duplicate data to another.


In [ ]:
df_cleaned = df.T.drop_duplicates(keep='first').T

df_cleaned.head()


In [ ]:
compare_df()
prev_df = df_cleaned

563 ---> 542 features

### Then, we consider what type of feature:
In the dataset, we've got two types of recorded data:
- t-* features: Raw time-series signals recorded at 50hz
- f-* features: Signals transformed using Fast Fourier Transform

There are mathematical functions that have been applied to numerous features, and in some cases the "t" and "p" version of these functions yield no new information. 


## Gemini breakdown because my dumbass dont know shit about physics:

### What you can safely remove:
f*-mean():
This function calculates the plain arithmetic mean of the Fourier amplitudes across all frequency bins. For primary acceleration, this is mathematically tied to the baseline time-domain average (tBodyAcc-mean()) or overall signal power. Dropping these across the board will cause almost zero loss in model accuracy because the time-domain equivalent (t) already captures the signal's overall magnitude

f*.max() + f*.min():
Why f is redundant/invalid: In the time domain, tBodyAcc-max()-X tells you the maximum G-force impulse experienced during a movement step. In the frequency domain, fBodyAcc-max()-X simply tells you the amplitude of the single strongest frequency peak. While peak frequency amplitude is informative, peak frequency intensity is already captured far more comprehensively by fBodyAcc-maxFreqInd (index of peak frequency) or fBodyAcc-energy()

f*.std() + f*.iqr() + f*.energy():
Why f is redundant/invalid: Thanks to Parseval’s Theorem, the total energy (variance) of a signal in the time domain is mathematically equal to the total energy of the signal in the frequency domain. Calculating dispersion statistics like std() or iqr() on the time signal tBodyAcc directly measures movement volatility. Doing the exact same calculation on the FFT bins (fBodyAcc-std()) simply measures how spread out the frequency amplitudes are, which strongly co-varies with time-domain variance.

Source for Parseval's Theorem and its valid application in sensory data feature extraction: 
 10.1109/access.2020.2996576



In [ ]:
total_features = df_cleaned.columns.tolist()

# Let's remove all f*-mean():
regex = r"^f.*-mean\(\)"
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

# Remove all f*-max():
regex = r"^f.*-max\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

# Remove all f*-min():
regex = r"^f.*-min\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)

df_cleaned.head()

In [ ]:
compare_df()
prev_df = df_cleaned

542 ---> 503 features

In [ ]:
# Remove all f*"-energy" (excluding bandsEnergy)
regex = r"^f.*-energy\(\)"
features_to_remove = []
features_to_remove = [i for i in total_features if re.search(regex, i)]
df_cleaned = df_cleaned.drop(columns=features_to_remove)
df_cleaned.head()

In [ ]:
compare_df()
prev_df = df_cleaned

542 ---> 490 features

# Variance filtering

We are to drop features that have x % sharing of same values.

In [ ]:
# We exclude 'subject' and 'Activity':
df_cleaned_no_categorical = df_cleaned.drop(columns=['Activity', 'subject'])

# We'll start easy: 1%.
selector = VarianceThreshold(threshold=0.01)
selector.fit(df_cleaned_no_categorical)

features_to_keep = df_cleaned_no_categorical.columns[selector.get_support()]

df_cleaned = df_cleaned[features_to_keep]

df_cleaned.head()

In [ ]:
compare_df()
prev_df = df_cleaned

### Working against multicollinearity
But if two features apply to this, which one to drop? We use Target-Aware Correlation Dropping:

In [ ]:
threshold = 0.95

corr_matrix = df_cleaned.corr().abs()

# These two segments are Gemini-generated:

# Extract upper triangle of correlation matrix (excluding diagonal)
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find features with correlation higher than threshold
to_drop = [
    column
    for column in upper_tri.columns
    if any(upper_tri[column] > threshold)
]

df_cleaned = df_cleaned.drop(columns = to_drop)


In [ ]:
compare_df()
prev_df = df_cleaned

### Bring back 'Activity' and 'subject'
Let's bring the gang back together babey

In [ ]:
# cleaned_df = pd.concat([df_cleaned, df['Activity', 'subject']], axis=1)